In [1]:
import numpy as np
from xgboost import XGBRegressor 
from sklearn.metrics import r2_score 
from data_processor import DataReader, DataPrep
from sklearn.model_selection import GridSearchCV
from cv_generator import train_val_split, ExpandingWindowCV
from sklearn.ensemble import RandomForestRegressor
from model_eval import *
import pandas as pd
from scipy import stats
from sklearn.preprocessing import LabelEncoder
from functools import partial
from lightgbm import LGBMRegressor
import warnings
import optuna
from sklearn.ensemble import RandomForestClassifier


In [2]:
import optuna

# You can use Matplotlib instead of Plotly for visualization by simply replacing `optuna.visualization` with
# `optuna.visualization.matplotlib` in the following examples.
from optuna.visualization import plot_contour
from optuna.visualization import plot_edf
from optuna.visualization import plot_intermediate_values
from optuna.visualization import plot_optimization_history
from optuna.visualization import plot_parallel_coordinate
from optuna.visualization import plot_param_importances
from optuna.visualization import plot_rank
from optuna.visualization import plot_slice
from optuna.visualization import plot_timeline

1. Clean the Dataprocessing

In [ ]:
daily = 'daily_df.pkl'
intraday = 'intraday_df.pkl'

# Check if the file exists in the current path

if not (os.path.exists(intraday) and os.path.exists(daily)):
    daily_data_path = r'data/daily_data'
    intraday_data_path = r'data/intraday_data'
    intraday_df = DataReader.read_intraday_data(intraday_data_path)
    daily_df = DataReader.read_daily_data(daily_data_path)
    daily_df.to_pickle('daily_df.pkl')
    intraday_df.to_pickle('intraday_df.pkl')

In [ ]:
daily_df = pd.read_pickle('daily_df.pkl')
intraday_df =  pd.read_pickle('intraday_df.pkl')
data_prep = DataPrep(intraday_df, daily_df)
target_df = data_prep.get_target(clip_MAD=True, normalize= True)
X_df = data_prep.get_features()#indicators= {'RSI_14':partial(ta.rsi, length= 14), 'RSI_5':partial(ta.rsi, length= 5)})
input_data = X_df.join(target_df[['y', 'y_actual_clipped']], how = 'right')

In [ ]:
X_df

### Train Val Split

In [ ]:
features = ['CumReturnResid',  'Rolling_Return_5d', 'Rolling_Return_10d', 'Rolling_Return_20d','NYSE', 'IntradayRSI']#, 'RSI_14']#,'Stock_Split', 'Dividend', 'Rolling_Return_20d', 'EarlyClose', 'NextHoliday']
clipped_returns = [col for col in input_data.columns if 'clipped' in col and 'Return' in col]
features = [f'Rolling_Return_{i}d_clipped' for i in [5, 10]]  + ['CumReturnResid', 'IntradayRSI', 'NYSE']# 'VolumeChangeNormalize']#, 'NYSE'] + 
input_data.dropna(subset=features, inplace=True)
train_data, val_data = train_val_split(input_data, 0.8)
print(features)
x_train, y_train = train_data[features], train_data['y']
x_val, y_val = val_data[features], val_data['y_actual_clipped']#val_data['y']
train_weights = train_data.MDV_63_sqrt.to_numpy()
val_weights = val_data.MDV_63_sqrt.to_numpy()

In [ ]:
train_data[clipped_returns + ['y']].corr()['y'].sort_values()

In [ ]:
train_data[features + ['y']].corr()['y'].sort_values()


### Base random forest grid search

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report
from sklearn.metrics import precision_recall_fscore_support
from sklearn.metrics import accuracy_score

In [ ]:
rf = RandomForestRegressor()
#TODO decomment
rf_param_grid = {
    'n_estimators': [100, 200, 300],
    'max_features': ['auto', 'sqrt', 'log2'],
    'max_depth': [2, 4, 6, 10],
    'min_samples_leaf': [2, 5, 10]
}

expand_wind_rf = ExpandingWindowCV(rf, rf_param_grid) 
expand_wind_rf.fit(x_train, y_train, train_weights)
print(expand_wind_rf.grid_search.best_params_)
model_rf = ModelEval(RandomForestRegressor(),features, x_train, y_train, train_weights, x_val.to_numpy(), y_val.to_numpy(), val_data.EST_VOL_preday.to_numpy(), val_weights,\
    grid_search= expand_wind_rf.grid_search)   
model_rf.to_pickle('xgboost_iter1')
print(model_rf.weighted_r2())
print(model_rf.feature_importance())
print(model_rf.feature_importance_MDA())

In [ ]:
x_train = x_train.drop(columns=x_train.columns[-1])

In [ ]:
x_val

In [ ]:
x_val = x_val.drop(columns=x_val.columns[-1])

In [ ]:
m = RandomForestRegressor()

In [ ]:
m.set_params

In [ ]:
y_train

In [ ]:
import numpy as np
from sklearn.model_selection import cross_validate, KFold
from sklearn.ensemble import RandomForestRegressor
import optuna
from functools import partial

def conduct_study(regressor, x_train, y_train, y_train_weights):
    
    def objective(trial):
        #n_estimators = trial.suggest_int("n_estimators", 100, 200, step=100)#300 not 200
        n_estimators = trial.suggest_int("n_estimators", 10, 30, step=10)
        max_depth = trial.suggest_int("max_depth", 2, 4, step=2) #2,10 NOT 2,4
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 2, step=1)# 10 not 2
    
        
        params = {
            "n_estimators": n_estimators,
            "max_depth": max_depth,
            "min_samples_leaf": min_samples_leaf
        }
        
        model = regressor.set_params(**params)
        
        cv_folds = ExpandingWindowCV._cv_index(x_train, 1)
        scores = []
        for train_index, test_index in cv_folds:
            X_train_fold, X_test_fold = x_train.iloc[train_index], x_train.iloc[test_index]
            y_train_fold, y_test_fold = y_train.iloc[train_index], y_train.iloc[test_index]
            y_train_weight, y_test_weight = y_train_weights[train_index], y_train_weights[test_index]
            
            model.fit(X_train_fold, y_train_fold, sample_weight=y_train_weight)
            y_pred = model.predict(X_test_fold)
            score = r2_score(y_test_fold,y_pred, sample_weight= y_test_weight)
            #score = model.score(X_test_fold, y_test_fold)  # Adjust scoring method as needed
            scores.append(score)
        
        mean_cv_score = np.mean(scores)
        return mean_cv_score

    study = optuna.create_study(direction='maximize')  # Or 'minimize', depending on the goal
    study.optimize(objective, n_trials=5)

    # After optimization, train a new model on the entire training set using the best parameters found
    best_params = study.best_params
    best_model = RandomForestRegressor(random_state=78, **best_params)
    best_model.fit(x_train, y_train, sample_weight=train_weights)
    
    return study

# Prediction and evaluation steps here
study = conduct_study(RandomForestRegressor(random_state= 78), x_train,y_train, train_weights)


In [ ]:
study.best_trial

In [ ]:
plot_optimization_history(study)

In [ ]:
plot_intermediate_values(study)

In [ ]:
plot_parallel_coordinate(study)

In [ ]:
plot_parallel_coordinate(study, params=["max_depth", "n_estimators"])

In [ ]:
plot_contour(study)

In [ ]:
plot_slice(study)

In [ ]:
plot_param_importances(study)

In [ ]:
len(set(x_train.index.get_level_values('Date')))

### Base xgboost Grid Search

In [ ]:
xgb = XGBRegressor()
xgb_param_grid = {
    'max_depth': [2, 4, 6],
    'learning_rate': [0.01, 0.1, 0.2],
    'n_estimators': [100, 200, 300],
    'subsample': [0.7, 0.9, 1.0],
    'colsample_bytree': [0.5, 0.7, 1.0],
    'min_child_weight': [1, 5, 10],
    'reg_alpha': [0, 0.1, 1],
    'reg_lambda': [1, 1.5, 2]
}

expand_wind_xgb = ExpandingWindowCV(xgb, xgb_param_grid) 
expand_wind_xgb.fit(x_train, y_train, train_weights)
print(expand_wind_xgb.grid_search.best_params_)
# best_param = {'colsample_bytree': 1.0, 'learning_rate': 0.01, 'max_depth': 2, 'min_child_weight': 1, 'n_estimators': 100, 'reg_alpha': 1, 'reg_lambda': 2, 'subsample': 0.7}
model = ModelEval(XGBRegressor(),features, x_train, y_train, train_weights, x_val.to_numpy(), y_val.to_numpy(), val_data.EST_VOL_preday.to_numpy(), val_weights, grid_search= expand_wind_xgb.grid_search)
# model.to_pickle('xgboost_base')
print(model.weighted_r2())
print(model.feature_importance())
print(model.feature_importance_MDA())

In [ ]:
val_data

### Base LightGBM Grid Search

In [ ]:
from lightgbm import LGBMRegressor

lgbm = LGBMRegressor(verbosity = -1)

lgbm_param_grid = {
    'max_depth': [2, 4, 6],
    'learning_rate': [0.01, 0.1, 0.2],
    'n_estimators': [100, 150],
    'subsample': [0.7, 0.9, 1.0],
    'colsample_bytree': [0.5, 0.7, 1.0],
    'min_child_samples': [1, 5, 10],
    'lambda_l2': [0.05, 0.1, 0.2, 0.3]
}

# expand_wind_lgbm = ExpandingWindowCV(lgbm, lgbm_param_grid) 
# expand_wind_lgbm.fit(x_train, y_train, train_weights)
# model_lgbm = ModelEval(LGBMRegressor(verbosity = -1),features, x_train, y_train, train_weights, x_val.to_numpy(), y_val.to_numpy(), val_data.EST_VOL_preday.to_numpy(), val_weights, expand_wind_lgbm.grid_search)
# model_lgbm.to_pickle('lgbm_iter2')
# model_lgbm = load_model('lgbm_iter1')
# print(model_lgbm.grid_cv.best_params_)
best_params = {'colsample_bytree': 1.0, 'lambda_l1': 0.3, 'learning_rate': 0.01, 'max_depth': 4, 'min_child_samples': 5, 'n_estimators': 100, 'subsample': 0.7}
model_lgbm = ModelEval(LGBMRegressor(verbosity = -1),features, x_train, y_train, train_weights, x_val.to_numpy(), y_val.to_numpy(), \
    val_data.EST_VOL_preday.to_numpy(), val_weights,best_params= best_params)
print(model_lgbm.weighted_r2())
print(model_lgbm.feature_importance())
print(model_lgbm.feature_importance_MDA())